# Predicción de Precios de Casas — Gradient Boosting (LightGBM + XGBoost)

## ¿Por qué Gradient Boosting en lugar de un modelo lineal?

Un modelo lineal asume que cada variable contribuye de forma **aditiva e independiente** al precio. Pero la realidad es más compleja:

- Una habitación extra vale mucho más en una casa de 200 m² que en una de 100 m²  
- Una calidad de cocina alta (`KitchenQual = Ex`) impacta más si la casa ya tiene buena calidad general  
- Un garaje para 3 coches en un vecindario premium vale más que en uno modesto

Estas son interacciones no lineales entre variables. Para capturarlas con regresión lineal necesitarías añadir manualmente todos los productos cruzados. Los árboles de decisión, en cambio, aprenden estas interacciones automáticamente en cada split.

LightGBM es una implementación de Gradient Boosting que:
- Entrena significativamente más rápido que XGBoost gracias a su algoritmo GOSS y EFB
- Maneja variables categóricas de forma nativa
- Requiere menos hiperparámetros que ajustar
- Es el estándar en competiciones de Kaggle para datos tabulares

## FASE A — Importaciones y carga de datos

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

import lightgbm as lgb
import xgboost as xgb

RANDOM_STATE = 42

# Variables seleccionadas (las más predictivas según el análisis previo)
winner_var_list = [
    "OverallQual", "GrLivArea", "TotalBsmtSF", "GarageCars",
    "YearBuilt", "Neighborhood", "BsmtQual", "KitchenQual"
]

# Variables categóricas donde NaN significa 'no tiene X' (no es dato faltante)
filter_meaning = [
    "Alley", "BsmtQual", "BsmtCond", "BsmtExposure",
    "BsmtFinType1", "BsmtFinType2", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "PoolQC", "Fence", "MiscFeature"
]

print("Librerías cargadas correctamente")
print(f"LightGBM versión: {lgb.__version__}")
print(f"XGBoost versión: {xgb.__version__}")

Librerías cargadas correctamente
LightGBM versión: 4.6.0
XGBoost versión: 3.2.0


In [3]:
dataset_df = pd.read_csv("train.csv")
print(f"Shape del dataset: {dataset_df.shape}")
dataset_df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

## FASE A — EDA (Análisis Exploratorio)

In [ ]:
# Distribución de la variable objetivo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(dataset_df["SalePrice"], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title("SalePrice — distribución original")
axes[0].set_xlabel("SalePrice")

axes[1].hist(np.log1p(dataset_df["SalePrice"]), bins=40, color='darkorange', edgecolor='white')
axes[1].set_title("log(SalePrice+1) — distribución normalizada")
axes[1].set_xlabel("log(SalePrice+1)")

plt.tight_layout()
plt.show()

# Razón: predecir en escala logarítmica es más justo: un error de 10k en una casa de 500k
# no debe penalizar igual que en una de 100k. RMSLE trata ambos proporcionalmente.

In [ ]:
# Correlación de las variables numéricas con SalePrice
num_cols = dataset_df[winner_var_list].select_dtypes(include=[np.number]).columns.tolist()

correlations = dataset_df[num_cols + ["SalePrice"]].corr()["SalePrice"].drop("SalePrice").sort_values(ascending=False)

plt.figure(figsize=(8, 4))
correlations.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title("Correlación lineal con SalePrice")
plt.ylabel("Pearson r")
plt.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# interacción GrLivArea x OverallQual → precio
# Esto es exactamente lo que un modelo lineal NO captura sin términos de interacción
plt.figure(figsize=(9, 5))
scatter = plt.scatter(
    x=dataset_df["GrLivArea"],
    y=dataset_df["SalePrice"],
    c=dataset_df["OverallQual"],
    cmap='RdYlGn',
    alpha=0.6,
    edgecolors='none',
    s=20
)
plt.colorbar(scatter, label='OverallQual')
plt.xlabel("GrLivArea (m² habitables)")
plt.ylabel("SalePrice")
plt.title("Interacción GrLivArea × OverallQual → SalePrice\n(color = calidad general)")
plt.tight_layout()
plt.show()

# Observación: para el mismo GrLivArea, el precio varía enormemente según OverallQual.
# Esto es una interacción que LightGBM aprende automáticamente.

## FASE B — Limpieza y preprocesado

In [ ]:
model_df = dataset_df[winner_var_list + ["SalePrice", "GrLivArea"]].copy()

# Eliminar outliers extremos en GrLivArea (>4500 sqft, casas atípicas)
# Estos puntos se alejan del patrón principal y pueden sesgar el modelo
n_before = len(model_df)
model_df = model_df[model_df["GrLivArea"] < 4500]
print(f"Filas eliminadas como outliers en GrLivArea: {n_before - len(model_df)}")
print(f"Filas restantes: {len(model_df)}")

In [ ]:
# ---- Gestión de NaN ----
# BsmtQual y KitchenQual: NaN significa 'sin sótano' → sustituir por 'None'
# TotalBsmtSF: NaN significa 0 pies cuadrados de sótano

model_df["BsmtQual"] = model_df["BsmtQual"].fillna("None")
model_df["KitchenQual"] = model_df["KitchenQual"].fillna("TA")   # moda
model_df["TotalBsmtSF"] = model_df["TotalBsmtSF"].fillna(0)
model_df["GarageCars"] = model_df["GarageCars"].fillna(0)

print("NaN restantes por columna:")
print(model_df.isna().sum())

In [ ]:
# ---- Encoding de variables categóricas ordinales ----
# BsmtQual y KitchenQual son ordinales: None < Po < Fa < TA < Gd < Ex
# Codificamos como enteros preservando el orden

qual_map = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}

model_df["BsmtQual_enc"]    = model_df["BsmtQual"].map(qual_map)
model_df["KitchenQual_enc"] = model_df["KitchenQual"].map(qual_map)

# Neighborhood: variable nominal → Label Encoding
# LightGBM puede tratar enteros como categóricos internamente, lo especificaremos después
le = LabelEncoder()
model_df["Neighborhood_enc"] = le.fit_transform(model_df["Neighborhood"])

print("Encoding completado")
print(f"Neighborhoods únicos: {model_df['Neighborhood'].nunique()}")

In [ ]:
# ---- Variable objetivo: transformación logarítmica ----
# Predecimos log(SalePrice+1) para reducir el impacto de casas caras
# y conseguir una distribución más simétrica (mejora el RMSE)

model_df["LogSalePrice"] = np.log1p(model_df["SalePrice"])

# ---- Features finales ----
feature_cols = [
    "OverallQual", "GrLivArea", "TotalBsmtSF", "GarageCars",
    "YearBuilt", "Neighborhood_enc", "BsmtQual_enc", "KitchenQual_enc"
]

X = model_df[feature_cols]
y = model_df["LogSalePrice"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape} | Validación: {X_val.shape}")

## FASE C — Ingeniería de Características

Aunque LightGBM captura interacciones automáticamente, añadir algunas features derivadas con sentido de negocio puede ayudar.

In [ ]:
def add_features(df):
    df = df.copy()
    
    # Edad de la casa en el momento de la venta (aprox.)
    df["HouseAge"] = 2010 - df["YearBuilt"]
    
    # Interacción explícita calidad × tamaño
    # (aunque LightGBM la descubriría sola, darla explícita acelera el aprendizaje)
    df["QualArea"] = df["OverallQual"] * df["GrLivArea"]
    
    # Ratio sótano / área habitable
    df["BsmtRatio"] = df["TotalBsmtSF"] / (df["GrLivArea"] + 1)
    
    return df

X_train_fe = add_features(X_train)
X_val_fe   = add_features(X_val)

feature_cols_fe = X_train_fe.columns.tolist()
print(f"Features con ingeniería: {feature_cols_fe}")

## FASE D — Modelado

### D1. LightGBM (modelo principal)

In [ ]:
# Construcción de los datasets de LightGBM
# categorical_feature: le decimos a LightGBM que Neighborhood_enc es categórica
# para que use su tratamiento especial (optimal split) en lugar de tratarla como numérica

lgb_train = lgb.Dataset(
    X_train_fe, label=y_train,
    categorical_feature=["Neighborhood_enc"],
    free_raw_data=False
)
lgb_val = lgb.Dataset(
    X_val_fe, label=y_val,
    categorical_feature=["Neighborhood_enc"],
    reference=lgb_train,
    free_raw_data=False
)

lgb_params = {
    "objective":        "regression",
    "metric":           "rmse",
    "learning_rate":    0.05,
    "num_leaves":       63,          # controla la complejidad del árbol
    "min_data_in_leaf": 20,          # evita sobreajuste en hojas pequeñas
    "feature_fraction": 0.8,         # usa el 80% de features en cada árbol (como Random Forest)
    "bagging_fraction": 0.8,         # usa el 80% de filas en cada árbol
    "bagging_freq":     5,
    "lambda_l1":        0.1,
    "lambda_l2":        0.1,
    "verbose":          -1,
    "random_state":     RANDOM_STATE,
}

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=False),
    lgb.log_evaluation(period=50)
]

lgb_model = lgb.train(
    lgb_params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_val],
    callbacks=callbacks
)

print(f"\nMejor iteración: {lgb_model.best_iteration}")

### D2. XGBoost (para comparación y posible ensamblado)

In [ ]:
xgb_params = {
    "objective":        "reg:squarederror",
    "eval_metric":      "rmse",
    "learning_rate":    0.05,
    "max_depth":        6,
    "min_child_weight": 5,
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "gamma":            0.1,
    "reg_alpha":        0.1,
    "reg_lambda":       0.1,
    "seed":             RANDOM_STATE,
    "verbosity":        0,
}

dtrain = xgb.DMatrix(X_train_fe, label=y_train)
dval   = xgb.DMatrix(X_val_fe,   label=y_val)

xgb_model = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=1000,
    evals=[(dval, "val")],
    early_stopping_rounds=50,
    verbose_eval=50
)

print(f"\nMejor iteración XGBoost: {xgb_model.best_iteration}")

## FASE E — Validación y comparativa

In [ ]:
def evaluate(y_true_log, y_pred_log, model_name="Modelo"):
    """Evalúa en escala logarítmica (RMSLE) y en escala original (MAE, R²)"""
    # Escala logarítmica
    rmsle = np.sqrt(mean_squared_error(y_true_log, y_pred_log))
    
    # Volver a escala original para interpretar en euros/dólares
    y_true_orig = np.expm1(y_true_log)
    y_pred_orig = np.expm1(y_pred_log)
    mae  = mean_absolute_error(y_true_orig, y_pred_orig)
    r2   = r2_score(y_true_orig, y_pred_orig)
    
    print(f"--- {model_name} ---")
    print(f"  RMSLE (log scale): {rmsle:.4f}")
    print(f"  MAE   (original):  ${mae:,.0f}")
    print(f"  R²    (original):   {r2:.4f}")
    return rmsle, mae, r2

# Predicciones
lgb_pred = lgb_model.predict(X_val_fe, num_iteration=lgb_model.best_iteration)
xgb_pred = xgb_model.predict(xgb.DMatrix(X_val_fe), iteration_range=(0, xgb_model.best_iteration))

# Ensamblado simple: media aritmética de ambos modelos
ensemble_pred = 0.6 * lgb_pred + 0.4 * xgb_pred

print("=" * 40)
lgb_scores  = evaluate(y_val, lgb_pred,       "LightGBM")
print()
xgb_scores  = evaluate(y_val, xgb_pred,       "XGBoost")
print()
ens_scores  = evaluate(y_val, ensemble_pred,  "Ensemble (LGB 60% + XGB 40%)")
print("=" * 40)

In [ ]:
# ---- Predicted vs Actual ----
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

y_true_orig = np.expm1(y_val.values)

for ax, (pred, name) in zip(axes, [
    (lgb_pred,      "LightGBM"),
    (xgb_pred,      "XGBoost"),
    (ensemble_pred, "Ensemble")
]):
    y_pred_orig = np.expm1(pred)
    ax.scatter(y_true_orig, y_pred_orig, alpha=0.4, s=15, color='steelblue')
    ax.plot([y_true_orig.min(), y_true_orig.max()],
            [y_true_orig.min(), y_true_orig.max()], 'r--', linewidth=1.5)
    ax.set_xlabel("Precio real")
    ax.set_ylabel("Precio predicho")
    ax.set_title(f"{name}\nR² = {r2_score(y_true_orig, y_pred_orig):.4f}")
    ax.ticklabel_format(style='sci', axis='both', scilimits=(0,0))

plt.tight_layout()
plt.show()

In [ ]:
# ---- Distribución de residuos ----
lgb_residuals = y_val.values - lgb_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(lgb_residuals, bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title("Distribución de residuos (LightGBM, log scale)")
axes[0].set_xlabel("Residuo")

axes[1].scatter(lgb_pred, lgb_residuals, alpha=0.4, s=15, color='darkorange')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title("Residuos vs Predicciones (LightGBM)")
axes[1].set_xlabel("Predicción (log scale)")
axes[1].set_ylabel("Residuo")

plt.tight_layout()
plt.show()

# Ideal: residuos centrados en 0 sin estructura → el modelo no tiene sesgo sistemático

In [ ]:
# ---- Importancia de variables (LightGBM) ----
# 'gain': reducción total de error aportada por cada variable (más informativa que 'split')

importance_df = pd.DataFrame({
    "feature":    lgb_model.feature_name(),
    "importance": lgb_model.feature_importance(importance_type="gain")
}).sort_values("importance", ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(importance_df["feature"], importance_df["importance"], color='steelblue')
plt.xlabel("Importancia (gain)")
plt.title("Importancia de variables — LightGBM")
plt.tight_layout()
plt.show()

print("\nVariables ordenadas por importancia:")
print(importance_df.sort_values('importance', ascending=False).to_string(index=False))

In [ ]:
# ---- Validación cruzada (K-Fold) para una estimación más robusta ----
# Evitamos el sesgo de un único split train/val

from sklearn.base import BaseEstimator, RegressorMixin

X_full = add_features(model_df[feature_cols])
y_full = model_df["LogSalePrice"]

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rmsle = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_full)):
    X_tr, X_vl = X_full.iloc[train_idx], X_full.iloc[val_idx]
    y_tr, y_vl = y_full.iloc[train_idx], y_full.iloc[val_idx]
    
    lgb_tr = lgb.Dataset(X_tr, label=y_tr, categorical_feature=["Neighborhood_enc"], free_raw_data=False)
    lgb_vl = lgb.Dataset(X_vl, label=y_vl, reference=lgb_tr, free_raw_data=False)
    
    m = lgb.train(
        lgb_params,
        lgb_tr,
        num_boost_round=lgb_model.best_iteration,
        valid_sets=[lgb_vl],
        callbacks=[lgb.log_evaluation(period=-1)]
    )
    preds = m.predict(X_vl)
    rmsle = np.sqrt(mean_squared_error(y_vl, preds))
    cv_rmsle.append(rmsle)
    print(f"Fold {fold+1}: RMSLE = {rmsle:.4f}")

print(f"\nCV RMSLE: {np.mean(cv_rmsle):.4f} ± {np.std(cv_rmsle):.4f}")

## FASE F — Conclusiones y próximos pasos

### Resultados obtenidos

| Modelo | RMSLE | R² |
|--------|-------|----|
| LightGBM | --- | ---- |
| XGBoost | ---- | ---- |
| Ensemble | mejor de ambos | mejor de ambos |

### ¿Por qué funciona mejor que un modelo lineal?

1. Captura interacciones automáticamente**: cada árbol aprende splits condicionados → "si GrLivArea > 1500 Y OverallQual > 7, entonces precio alto"
2. No asume linealidad: la relación entre YearBuilt y precio no es lineal (casas muy antiguas o muy nuevas tienen dinámicas distintas)
3. Maneja variables categóricas ordinales: BsmtQual y KitchenQual se tratan como lo que son
4. Robusto a outliers: los splits acotan el efecto de valores extremos

